In [51]:
import sys
from pathlib import Path
# Add the parent directory to sys.path so we can import synth_extract
sys.path.insert(0, str(Path.cwd().parent)) 

In [52]:
import importlib 
# importlib.reload(sys.modules['synth_extract.utils.markdown_helpers'])
from synth_extract.utils.markdown_helpers import pdf_to_markdown, pdf_to_markdown_using_cli

Testing

In [ ]:
pdf_path = Path("../../pdfs")
pdf_files = list(pdf_path.glob("*.pdf"))

In [ ]:
pdf_files[5]

In [ ]:
markdown_path = Path("../../pdf_library")

In [ ]:
pdf_to_markdown(pdf_files[5], markdown_path, markdown_only=True)

Testing on dev set

In [ ]:
base_path = Path("../data/development_set")
arxiv_path = base_path / "arxiv"
wiley_path = base_path / "wiley"

In [ ]:
# Convert each arXiv and Wiley PDF to Markdown beside the source PDF
converted = 0
missing_pdf = []
failed = []

for source_path in (arxiv_path, wiley_path):
    if not source_path.is_dir():
        print(f"MISSING SOURCE DIRECTORY: {source_path}")
        continue

    paper_dirs = sorted(path for path in source_path.iterdir() if path.is_dir())
    for index, paper_dir in enumerate(paper_dirs, start=1):
        pdf_files = sorted(
            path
            for path in paper_dir.iterdir()
            if path.is_file() and path.suffix.lower() == ".pdf"
        )

        if not pdf_files:
            missing_pdf.append(paper_dir)
            print(f"NO PDF: {paper_dir}")
            continue

        for pdf_file in pdf_files:
            print(
                f"[{source_path.name} {index}/{len(paper_dirs)}] "
                f"Converting {pdf_file}"
            )
            try:
                pdf_to_markdown(pdf_file, markdown_only=True)
                converted += 1
            except Exception as exc:
                failed.append((pdf_file, exc))
                print(f"FAILED: {pdf_file}: {exc}")

print(f"Converted PDFs: {converted:,}")
print(f"Paper folders without a PDF: {len(missing_pdf):,}")
print(f"Failed conversions: {len(failed):,}")

creating track dbs

In [ ]:
# # Build the ArXiv Markdown-conversion tracking database.
# from contextlib import closing
# from pathlib import Path
# import sqlite3

# central_db_path = Path("../data/central_papers.db").resolve()
# track_db_path = Path("../data/process/arxiv_track.db").resolve()

# if not central_db_path.is_file():
#     raise FileNotFoundError(f"Central database not found: {central_db_path}")
# track_db_path.parent.mkdir(parents=True, exist_ok=True)

# central_uri = f"{central_db_path.as_uri()}?mode=ro"
# with closing(sqlite3.connect(central_uri, uri=True)) as source_conn:
#     arxiv_rows = source_conn.execute(
#         """
#         SELECT paper_id, paper_uid, canonical_source
#         FROM papers
#         WHERE canonical_source = 'arxiv'
#           AND download_status = 'success'
#         ORDER BY paper_id
#         """
#     ).fetchall()

# source_count = len(arxiv_rows)

# with closing(
#     sqlite3.connect(track_db_path, timeout=60, isolation_level=None)
# ) as conn:
#     conn.execute("PRAGMA busy_timeout = 60000")

#     try:
#         conn.execute("BEGIN IMMEDIATE")

#         # Rebuild the tracking table so every conversion starts unprocessed.
#         conn.execute("DROP TABLE IF EXISTS papers")
#         conn.execute(
#             """
#             CREATE TABLE papers (
#                 paper_id INTEGER PRIMARY KEY,
#                 paper_uid TEXT NOT NULL UNIQUE,
#                 canonical_source TEXT NOT NULL,
#                 convert_md INTEGER DEFAULT NULL
#             )
#             """
#         )
#         conn.executemany(
#             """
#             INSERT INTO papers (
#                 paper_id, paper_uid, canonical_source, convert_md
#             )
#             VALUES (?, ?, ?, NULL)
#             """,
#             arxiv_rows,
#         )

#         copied_count = conn.execute(
#             "SELECT COUNT(*) FROM papers"
#         ).fetchone()[0]
#         non_null_progress_count = conn.execute(
#             "SELECT COUNT(*) FROM papers WHERE convert_md IS NOT NULL"
#         ).fetchone()[0]

#         if copied_count != source_count:
#             raise RuntimeError(
#                 f"Expected to copy {source_count:,} rows, "
#                 f"but copied {copied_count:,}"
#             )
#         if non_null_progress_count:
#             raise RuntimeError(
#                 f"{non_null_progress_count:,} convert_md value(s) "
#                 "were unexpectedly non-NULL"
#             )

#         conn.commit()
#     except Exception:
#         conn.rollback()
#         raise

# print(f"Created {track_db_path}")
# print(f"ArXiv rows copied: {copied_count:,}")

Created /Users/kevinge/Work/Data Extraction/synth_extract/data/process/arxiv_track.db
ArXiv rows copied: 5,388


In [ ]:
# # Build the Elsevier Markdown-conversion tracking database.
# from contextlib import closing
# from pathlib import Path
# import sqlite3

# central_db_path = Path("../data/central_papers.db").resolve()
# track_db_path = Path("../data/process/elsevier_track.db").resolve()

# if not central_db_path.is_file():
#     raise FileNotFoundError(f"Central database not found: {central_db_path}")
# track_db_path.parent.mkdir(parents=True, exist_ok=True)

# central_uri = f"{central_db_path.as_uri()}?mode=ro"
# with closing(sqlite3.connect(central_uri, uri=True)) as source_conn:
#     elsevier_rows = source_conn.execute(
#         """
#         SELECT paper_id, paper_uid, canonical_source
#         FROM papers
#         WHERE canonical_source = 'elsevier'
#           AND download_status = 'success'
#         ORDER BY paper_id
#         """
#     ).fetchall()

# source_count = len(elsevier_rows)

# with closing(
#     sqlite3.connect(track_db_path, timeout=60, isolation_level=None)
# ) as conn:
#     conn.execute("PRAGMA busy_timeout = 60000")

#     try:
#         conn.execute("BEGIN IMMEDIATE")

#         # Rebuild the tracking table so every conversion starts unprocessed.
#         conn.execute("DROP TABLE IF EXISTS papers")
#         conn.execute(
#             """
#             CREATE TABLE papers (
#                 paper_id INTEGER PRIMARY KEY,
#                 paper_uid TEXT NOT NULL UNIQUE,
#                 canonical_source TEXT NOT NULL,
#                 convert_md INTEGER DEFAULT NULL
#             )
#             """
#         )
#         conn.executemany(
#             """
#             INSERT INTO papers (
#                 paper_id, paper_uid, canonical_source, convert_md
#             )
#             VALUES (?, ?, ?, NULL)
#             """,
#             elsevier_rows,
#         )

#         copied_count = conn.execute(
#             "SELECT COUNT(*) FROM papers"
#         ).fetchone()[0]
#         non_null_progress_count = conn.execute(
#             "SELECT COUNT(*) FROM papers WHERE convert_md IS NOT NULL"
#         ).fetchone()[0]

#         if copied_count != source_count:
#             raise RuntimeError(
#                 f"Expected to copy {source_count:,} rows, "
#                 f"but copied {copied_count:,}"
#             )
#         if non_null_progress_count:
#             raise RuntimeError(
#                 f"{non_null_progress_count:,} convert_md value(s) "
#                 "were unexpectedly non-NULL"
#             )

#         conn.commit()
#     except Exception:
#         conn.rollback()
#         raise

# print(f"Created {track_db_path}")
# print(f"Successful Elsevier rows copied: {copied_count:,}")

Created /Users/kevinge/Work/Data Extraction/synth_extract/data/process/elsevier_track.db
Successful Elsevier rows copied: 489,472


In [ ]:
# # Build the Europe PMC Markdown-conversion tracking database.
# from contextlib import closing
# from pathlib import Path
# import sqlite3

# central_db_path = Path("../data/central_papers.db").resolve()
# track_db_path = Path("../data/process/europepmc_track.db").resolve()

# if not central_db_path.is_file():
#     raise FileNotFoundError(f"Central database not found: {central_db_path}")
# track_db_path.parent.mkdir(parents=True, exist_ok=True)

# central_uri = f"{central_db_path.as_uri()}?mode=ro"
# with closing(sqlite3.connect(central_uri, uri=True)) as source_conn:
#     europepmc_rows = source_conn.execute(
#         """
#         SELECT paper_id, paper_uid, canonical_source
#         FROM papers
#         WHERE canonical_source = 'europepmc'
#           AND download_status = 'success'
#         ORDER BY paper_id
#         """
#     ).fetchall()

# source_count = len(europepmc_rows)

# with closing(
#     sqlite3.connect(track_db_path, timeout=60, isolation_level=None)
# ) as conn:
#     conn.execute("PRAGMA busy_timeout = 60000")

#     try:
#         conn.execute("BEGIN IMMEDIATE")

#         # Rebuild the tracking table so every conversion starts unprocessed.
#         conn.execute("DROP TABLE IF EXISTS papers")
#         conn.execute(
#             """
#             CREATE TABLE papers (
#                 paper_id INTEGER PRIMARY KEY,
#                 paper_uid TEXT NOT NULL UNIQUE,
#                 canonical_source TEXT NOT NULL,
#                 convert_md INTEGER DEFAULT NULL
#             )
#             """
#         )
#         conn.executemany(
#             """
#             INSERT INTO papers (
#                 paper_id, paper_uid, canonical_source, convert_md
#             )
#             VALUES (?, ?, ?, NULL)
#             """,
#             europepmc_rows,
#         )

#         copied_count = conn.execute(
#             "SELECT COUNT(*) FROM papers"
#         ).fetchone()[0]
#         non_null_progress_count = conn.execute(
#             "SELECT COUNT(*) FROM papers WHERE convert_md IS NOT NULL"
#         ).fetchone()[0]

#         if copied_count != source_count:
#             raise RuntimeError(
#                 f"Expected to copy {source_count:,} rows, "
#                 f"but copied {copied_count:,}"
#             )
#         if non_null_progress_count:
#             raise RuntimeError(
#                 f"{non_null_progress_count:,} convert_md value(s) "
#                 "were unexpectedly non-NULL"
#             )

#         conn.commit()
#     except Exception:
#         conn.rollback()
#         raise

# print(f"Created {track_db_path}")
# print(f"Successful Europe PMC rows copied: {copied_count:,}")

In [ ]:
# # Build the S2ORC Markdown-conversion tracking database.
# from contextlib import closing
# from pathlib import Path
# import sqlite3

# central_db_path = Path("../data/central_papers.db").resolve()
# track_db_path = Path("../data/process/s2orc_track.db").resolve()

# if not central_db_path.is_file():
#     raise FileNotFoundError(f"Central database not found: {central_db_path}")
# track_db_path.parent.mkdir(parents=True, exist_ok=True)

# central_uri = f"{central_db_path.as_uri()}?mode=ro"
# with closing(sqlite3.connect(central_uri, uri=True)) as source_conn:
#     s2orc_rows = source_conn.execute(
#         """
#         SELECT paper_id, paper_uid, canonical_source
#         FROM papers
#         WHERE canonical_source = 's2orc'
#           AND download_status = 'success'
#         ORDER BY paper_id
#         """
#     ).fetchall()

# source_count = len(s2orc_rows)

# with closing(
#     sqlite3.connect(track_db_path, timeout=60, isolation_level=None)
# ) as conn:
#     conn.execute("PRAGMA busy_timeout = 60000")

#     try:
#         conn.execute("BEGIN IMMEDIATE")

#         # Rebuild the tracking table so every conversion starts unprocessed.
#         conn.execute("DROP TABLE IF EXISTS papers")
#         conn.execute(
#             """
#             CREATE TABLE papers (
#                 paper_id INTEGER PRIMARY KEY,
#                 paper_uid TEXT NOT NULL UNIQUE,
#                 canonical_source TEXT NOT NULL,
#                 convert_md INTEGER DEFAULT NULL
#             )
#             """
#         )
#         conn.executemany(
#             """
#             INSERT INTO papers (
#                 paper_id, paper_uid, canonical_source, convert_md
#             )
#             VALUES (?, ?, ?, NULL)
#             """,
#             s2orc_rows,
#         )

#         copied_count = conn.execute(
#             "SELECT COUNT(*) FROM papers"
#         ).fetchone()[0]
#         non_null_progress_count = conn.execute(
#             "SELECT COUNT(*) FROM papers WHERE convert_md IS NOT NULL"
#         ).fetchone()[0]

#         if copied_count != source_count:
#             raise RuntimeError(
#                 f"Expected to copy {source_count:,} rows, "
#                 f"but copied {copied_count:,}"
#             )
#         if non_null_progress_count:
#             raise RuntimeError(
#                 f"{non_null_progress_count:,} convert_md value(s) "
#                 "were unexpectedly non-NULL"
#             )

#         conn.commit()
#     except Exception:
#         conn.rollback()
#         raise

# print(f"Created {track_db_path}")
# print(f"Successful S2ORC rows copied: {copied_count:,}")

Created /Users/kevinge/Work/Data Extraction/synth_extract/data/process/s2orc_track.db
Successful S2ORC rows copied: 150,105


In [ ]:
# # Build the Springer Nature Markdown-conversion tracking database.
# from contextlib import closing
# from pathlib import Path
# import sqlite3

# central_db_path = Path("../data/central_papers.db").resolve()
# track_db_path = Path("../data/process/springer_nature_track.db").resolve()

# if not central_db_path.is_file():
#     raise FileNotFoundError(f"Central database not found: {central_db_path}")
# track_db_path.parent.mkdir(parents=True, exist_ok=True)

# central_uri = f"{central_db_path.as_uri()}?mode=ro"
# with closing(sqlite3.connect(central_uri, uri=True)) as source_conn:
#     springer_nature_rows = source_conn.execute(
#         """
#         SELECT paper_id, paper_uid, canonical_source
#         FROM papers
#         WHERE canonical_source = 'springer_nature'
#           AND download_status = 'success'
#         ORDER BY paper_id
#         """
#     ).fetchall()

# source_count = len(springer_nature_rows)

# with closing(
#     sqlite3.connect(track_db_path, timeout=60, isolation_level=None)
# ) as conn:
#     conn.execute("PRAGMA busy_timeout = 60000")

#     try:
#         conn.execute("BEGIN IMMEDIATE")

#         # Rebuild the tracking table so every conversion starts unprocessed.
#         conn.execute("DROP TABLE IF EXISTS papers")
#         conn.execute(
#             """
#             CREATE TABLE papers (
#                 paper_id INTEGER PRIMARY KEY,
#                 paper_uid TEXT NOT NULL UNIQUE,
#                 canonical_source TEXT NOT NULL,
#                 convert_md INTEGER DEFAULT NULL
#             )
#             """
#         )
#         conn.executemany(
#             """
#             INSERT INTO papers (
#                 paper_id, paper_uid, canonical_source, convert_md
#             )
#             VALUES (?, ?, ?, NULL)
#             """,
#             springer_nature_rows,
#         )

#         copied_count = conn.execute(
#             "SELECT COUNT(*) FROM papers"
#         ).fetchone()[0]
#         non_null_progress_count = conn.execute(
#             "SELECT COUNT(*) FROM papers WHERE convert_md IS NOT NULL"
#         ).fetchone()[0]

#         if copied_count != source_count:
#             raise RuntimeError(
#                 f"Expected to copy {source_count:,} rows, "
#                 f"but copied {copied_count:,}"
#             )
#         if non_null_progress_count:
#             raise RuntimeError(
#                 f"{non_null_progress_count:,} convert_md value(s) "
#                 "were unexpectedly non-NULL"
#             )

#         conn.commit()
#     except Exception:
#         conn.rollback()
#         raise

# print(f"Created {track_db_path}")
# print(f"Successful Springer Nature rows copied: {copied_count:,}")

Created /Users/kevinge/Work/Data Extraction/synth_extract/data/process/springer_nature_track.db
Successful Springer Nature rows copied: 6,103


In [ ]:
# # Build the Wiley Markdown-conversion tracking database.
# from contextlib import closing
# from pathlib import Path
# import sqlite3

# central_db_path = Path("../data/central_papers.db").resolve()
# track_db_path = Path("../data/process/wiley_track.db").resolve()

# if not central_db_path.is_file():
#     raise FileNotFoundError(f"Central database not found: {central_db_path}")
# track_db_path.parent.mkdir(parents=True, exist_ok=True)

# central_uri = f"{central_db_path.as_uri()}?mode=ro"
# with closing(sqlite3.connect(central_uri, uri=True)) as source_conn:
#     wiley_rows = source_conn.execute(
#         """
#         SELECT paper_id, paper_uid, canonical_source
#         FROM papers
#         WHERE canonical_source = 'wiley'
#           AND download_status = 'success'
#         ORDER BY paper_id
#         """
#     ).fetchall()

# source_count = len(wiley_rows)

# with closing(
#     sqlite3.connect(track_db_path, timeout=60, isolation_level=None)
# ) as conn:
#     conn.execute("PRAGMA busy_timeout = 60000")

#     try:
#         conn.execute("BEGIN IMMEDIATE")

#         # Rebuild the tracking table so every conversion starts unprocessed.
#         conn.execute("DROP TABLE IF EXISTS papers")
#         conn.execute(
#             """
#             CREATE TABLE papers (
#                 paper_id INTEGER PRIMARY KEY,
#                 paper_uid TEXT NOT NULL UNIQUE,
#                 canonical_source TEXT NOT NULL,
#                 convert_md INTEGER DEFAULT NULL
#             )
#             """
#         )
#         conn.executemany(
#             """
#             INSERT INTO papers (
#                 paper_id, paper_uid, canonical_source, convert_md
#             )
#             VALUES (?, ?, ?, NULL)
#             """,
#             wiley_rows,
#         )

#         copied_count = conn.execute(
#             "SELECT COUNT(*) FROM papers"
#         ).fetchone()[0]
#         non_null_progress_count = conn.execute(
#             "SELECT COUNT(*) FROM papers WHERE convert_md IS NOT NULL"
#         ).fetchone()[0]

#         if copied_count != source_count:
#             raise RuntimeError(
#                 f"Expected to copy {source_count:,} rows, "
#                 f"but copied {copied_count:,}"
#             )
#         if non_null_progress_count:
#             raise RuntimeError(
#                 f"{non_null_progress_count:,} convert_md value(s) "
#                 "were unexpectedly non-NULL"
#             )

#         conn.commit()
#     except Exception:
#         conn.rollback()
#         raise

# print(f"Created {track_db_path}")
# print(f"Successful Wiley rows copied: {copied_count:,}")

Created /Users/kevinge/Work/Data Extraction/synth_extract/data/process/wiley_track.db
Successful Wiley rows copied: 176,784


creating dummies

In [ ]:
# # Create dummy tracking databases from the development-set folders.
# from contextlib import closing
# from pathlib import Path
# import re
# import sqlite3

# development_root = Path("../data/development_set").resolve()
# uid_pattern = re.compile(r"^ID(\d+)$")
# source_pattern = re.compile(r"^[A-Za-z0-9_]+$")

# if not development_root.is_dir():
#     raise FileNotFoundError(
#         f"Development-set directory not found: {development_root}"
#     )

# # Discover and validate every source and paper folder before writing a DB.
# source_rows = {}
# for source_dir in sorted(
#     path for path in development_root.iterdir() if path.is_dir()
# ):
#     source = source_dir.name
#     if not source_pattern.fullmatch(source):
#         raise ValueError(f"Unsafe source folder name: {source!r}")

#     rows = []
#     for paper_dir in sorted(
#         path for path in source_dir.iterdir() if path.is_dir()
#     ):
#         paper_uid = paper_dir.name
#         match = uid_pattern.fullmatch(paper_uid)
#         if match is None:
#             raise ValueError(
#                 f"Invalid paper folder in {source}: {paper_uid!r}"
#             )

#         paper_id = int(match.group(1))
#         rows.append((paper_id, paper_uid, source))

#     if not rows:
#         raise ValueError(f"No paper folders found in {source_dir}")
#     if len({row[0] for row in rows}) != len(rows):
#         raise ValueError(f"Duplicate paper IDs found in {source_dir}")
#     if len({row[1] for row in rows}) != len(rows):
#         raise ValueError(f"Duplicate paper UIDs found in {source_dir}")

#     source_rows[source] = rows

# creation_summary = []
# for source, rows in source_rows.items():
#     track_db_path = development_root / f"{source}_track.db"

#     with closing(
#         sqlite3.connect(track_db_path, timeout=60, isolation_level=None)
#     ) as conn:
#         conn.execute("PRAGMA busy_timeout = 60000")

#         try:
#             conn.execute("BEGIN IMMEDIATE")
#             conn.execute("DROP TABLE IF EXISTS papers")
#             conn.execute(
#                 """
#                 CREATE TABLE papers (
#                     paper_id INTEGER PRIMARY KEY,
#                     paper_uid TEXT NOT NULL UNIQUE,
#                     canonical_source TEXT NOT NULL,
#                     convert_md INTEGER DEFAULT NULL
#                 )
#                 """
#             )
#             conn.executemany(
#                 """
#                 INSERT INTO papers (
#                     paper_id, paper_uid, canonical_source, convert_md
#                 )
#                 VALUES (?, ?, ?, NULL)
#                 """,
#                 rows,
#             )

#             copied_count = conn.execute(
#                 "SELECT COUNT(*) FROM papers"
#             ).fetchone()[0]
#             pending_count = conn.execute(
#                 "SELECT COUNT(*) FROM papers WHERE convert_md IS NULL"
#             ).fetchone()[0]
#             if copied_count != len(rows) or pending_count != len(rows):
#                 raise RuntimeError(
#                     f"Verification failed for {track_db_path}"
#                 )

#             conn.commit()
#         except Exception:
#             conn.rollback()
#             raise

#     creation_summary.append((source, copied_count, track_db_path))

# for source, row_count, track_db_path in creation_summary:
#     print(f"{source}: {row_count:,} rows -> {track_db_path}")

arxiv: 20 rows -> /Users/kevinge/Work/Data Extraction/synth_extract/data/development_set/arxiv_track.db
elsevier: 20 rows -> /Users/kevinge/Work/Data Extraction/synth_extract/data/development_set/elsevier_track.db
europepmc: 20 rows -> /Users/kevinge/Work/Data Extraction/synth_extract/data/development_set/europepmc_track.db
s2orc: 20 rows -> /Users/kevinge/Work/Data Extraction/synth_extract/data/development_set/s2orc_track.db
springer_nature: 20 rows -> /Users/kevinge/Work/Data Extraction/synth_extract/data/development_set/springer_nature_track.db
wiley: 20 rows -> /Users/kevinge/Work/Data Extraction/synth_extract/data/development_set/wiley_track.db


In [ ]:
# # Split the Wiley tracking database into parts of at most 5,000 rows.
# from contextlib import closing
# from pathlib import Path
# import sqlite3

# process_dir = Path("../data/process").resolve()
# source_db_path = process_dir / "wiley_track.db"
# rows_per_part = 5_000

# if not source_db_path.is_file():
#     raise FileNotFoundError(
#         f"Wiley tracking database not found: {source_db_path}"
#     )
# if rows_per_part < 1:
#     raise ValueError("rows_per_part must be at least 1")

# source_uri = f"{source_db_path.as_uri()}?mode=ro"
# created_parts = []

# with closing(sqlite3.connect(source_uri, uri=True)) as source_conn:
#     source_conn.execute("PRAGMA query_only = ON")

#     table_exists = source_conn.execute(
#         """
#         SELECT 1
#         FROM sqlite_master
#         WHERE type = 'table' AND name = 'papers'
#         """
#     ).fetchone()
#     if table_exists is None:
#         raise ValueError("Source database has no papers table")

#     source_columns = {
#         row[1] for row in source_conn.execute(
#             "PRAGMA table_info(papers)"
#         )
#     }
#     required_columns = {
#         "paper_id",
#         "paper_uid",
#         "canonical_source",
#         "convert_md",
#     }
#     missing_columns = sorted(required_columns - source_columns)
#     if missing_columns:
#         raise ValueError(
#             "Source papers table is missing: "
#             + ", ".join(missing_columns)
#         )

#     total_rows = source_conn.execute(
#         "SELECT COUNT(*) FROM papers"
#     ).fetchone()[0]
#     if total_rows == 0:
#         raise ValueError("Wiley tracking database contains no rows")

#     expected_part_count = (total_rows + rows_per_part - 1) // rows_per_part
#     expected_paths = {
#         process_dir / f"wiley_track_part_{part_number}.db"
#         for part_number in range(1, expected_part_count + 1)
#     }
#     stale_paths = (
#         set(process_dir.glob("wiley_track_part_*.db"))
#         - expected_paths
#     )
#     if stale_paths:
#         stale_names = ", ".join(
#             path.name for path in sorted(stale_paths)
#         )
#         raise RuntimeError(
#             "Stale Wiley part database(s) must be moved or removed: "
#             + stale_names
#         )

#     # Hold a consistent read snapshot while all parts are created.
#     source_conn.execute("BEGIN")
#     last_paper_id = None
#     copied_total = 0

#     for part_number in range(1, expected_part_count + 1):
#         if last_paper_id is None:
#             rows = source_conn.execute(
#                 """
#                 SELECT paper_id, paper_uid, canonical_source, convert_md
#                 FROM papers
#                 ORDER BY paper_id
#                 LIMIT ?
#                 """,
#                 (rows_per_part,),
#             ).fetchall()
#         else:
#             rows = source_conn.execute(
#                 """
#                 SELECT paper_id, paper_uid, canonical_source, convert_md
#                 FROM papers
#                 WHERE paper_id > ?
#                 ORDER BY paper_id
#                 LIMIT ?
#                 """,
#                 (last_paper_id, rows_per_part),
#             ).fetchall()

#         if not rows:
#             raise RuntimeError(
#                 f"Part {part_number} unexpectedly contains no rows"
#             )

#         part_path = process_dir / (
#             f"wiley_track_part_{part_number}.db"
#         )
#         temporary_path = part_path.with_suffix(".db.tmp")
#         if temporary_path.exists():
#             temporary_path.unlink()

#         try:
#             with closing(
#                 sqlite3.connect(
#                     temporary_path, timeout=60, isolation_level=None
#                 )
#             ) as part_conn:
#                 part_conn.execute("PRAGMA busy_timeout = 60000")
#                 part_conn.execute("BEGIN IMMEDIATE")
#                 part_conn.execute(
#                     """
#                     CREATE TABLE papers (
#                         paper_id INTEGER PRIMARY KEY,
#                         paper_uid TEXT NOT NULL UNIQUE,
#                         canonical_source TEXT NOT NULL,
#                         convert_md INTEGER DEFAULT NULL
#                     )
#                     """
#                 )
#                 part_conn.executemany(
#                     """
#                     INSERT INTO papers (
#                         paper_id, paper_uid, canonical_source, convert_md
#                     )
#                     VALUES (?, ?, ?, ?)
#                     """,
#                     rows,
#                 )

#                 part_count = part_conn.execute(
#                     "SELECT COUNT(*) FROM papers"
#                 ).fetchone()[0]
#                 if part_count != len(rows) or part_count > rows_per_part:
#                     raise RuntimeError(
#                         f"Verification failed for {part_path.name}"
#                     )
#                 part_conn.commit()

#             # Replace an earlier version only after the new part is valid.
#             temporary_path.replace(part_path)
#         except Exception:
#             if temporary_path.exists():
#                 temporary_path.unlink()
#             raise

#         last_paper_id = rows[-1][0]
#         copied_total += len(rows)
#         created_parts.append((part_number, len(rows), part_path))

#     source_conn.rollback()

# if copied_total != total_rows:
#     raise RuntimeError(
#         f"Expected {total_rows:,} rows but copied {copied_total:,}"
#     )

# print(
#     f"Split {total_rows:,} Wiley rows into "
#     f"{len(created_parts)} database parts."
# )
# for part_number, row_count, part_path in created_parts:
#     print(f"Part {part_number:>2}: {row_count:,} rows -> {part_path}")


Split 176,784 Wiley rows into 36 database parts.
Part  1: 5,000 rows -> /Users/kevinge/Work/Data Extraction/synth_extract/data/process/wiley_track_part_1.db
Part  2: 5,000 rows -> /Users/kevinge/Work/Data Extraction/synth_extract/data/process/wiley_track_part_2.db
Part  3: 5,000 rows -> /Users/kevinge/Work/Data Extraction/synth_extract/data/process/wiley_track_part_3.db
Part  4: 5,000 rows -> /Users/kevinge/Work/Data Extraction/synth_extract/data/process/wiley_track_part_4.db
Part  5: 5,000 rows -> /Users/kevinge/Work/Data Extraction/synth_extract/data/process/wiley_track_part_5.db
Part  6: 5,000 rows -> /Users/kevinge/Work/Data Extraction/synth_extract/data/process/wiley_track_part_6.db
Part  7: 5,000 rows -> /Users/kevinge/Work/Data Extraction/synth_extract/data/process/wiley_track_part_7.db
Part  8: 5,000 rows -> /Users/kevinge/Work/Data Extraction/synth_extract/data/process/wiley_track_part_8.db
Part  9: 5,000 rows -> /Users/kevinge/Work/Data Extraction/synth_extract/data/process/w

#### Markdown conversion progress

In [45]:
# Summarize Markdown-conversion progress from the tracking databases.
from contextlib import closing
from pathlib import Path
import re
import sqlite3

import pandas as pd

process_dir = Path("../data/process").resolve()
main_sources = [
    "arxiv",
    "elsevier",
    "europepmc",
    "s2orc",
    "springer_nature",
]

if not process_dir.is_dir():
    raise FileNotFoundError(f"Process directory not found: {process_dir}")

database_paths = {
    source: [process_dir / f"{source}_track.db"]
    for source in main_sources
}

# Wiley progress lives in numbered parts; do not use wiley_track.db.
wiley_part_pattern = re.compile(r"^wiley_track_part_(\d+)\.db$")
wiley_parts = []
for path in process_dir.glob("wiley_track_part_*.db"):
    match = wiley_part_pattern.fullmatch(path.name)
    if match is not None:
        wiley_parts.append((int(match.group(1)), path))

wiley_parts.sort(key=lambda item: item[0])
if not wiley_parts:
    raise FileNotFoundError(
        f"No Wiley part databases found in {process_dir}"
    )

part_numbers = [part_number for part_number, _ in wiley_parts]
expected_part_numbers = list(range(1, part_numbers[-1] + 1))
if part_numbers != expected_part_numbers:
    missing_parts = sorted(set(expected_part_numbers) - set(part_numbers))
    raise RuntimeError(f"Missing Wiley database part(s): {missing_parts}")

database_paths["wiley"] = [path for _, path in wiley_parts]

summary_rows = []
for source, paths in database_paths.items():
    total_papers = 0
    successful_markdown = 0

    for db_path in paths:
        if not db_path.is_file():
            raise FileNotFoundError(
                f"Tracking database not found: {db_path}"
            )

        database_uri = f"{db_path.as_uri()}?mode=ro"
        with closing(
            sqlite3.connect(database_uri, uri=True, timeout=60)
        ) as conn:
            conn.execute("PRAGMA query_only = ON")
            conn.execute("PRAGMA busy_timeout = 60000")

            table_exists = conn.execute(
                """
                SELECT 1
                FROM sqlite_master
                WHERE type = 'table' AND name = 'papers'
                """
            ).fetchone()
            if table_exists is None:
                raise ValueError(f"No papers table in {db_path}")

            columns = {
                row[1] for row in conn.execute(
                    "PRAGMA table_info(papers)"
                )
            }
            if "convert_md" not in columns:
                raise ValueError(
                    f"No convert_md column in {db_path}"
                )

            db_total, db_successful = conn.execute(
                """
                SELECT
                    COUNT(*),
                    COALESCE(
                        SUM(CASE WHEN convert_md = 1 THEN 1 ELSE 0 END),
                        0
                    )
                FROM papers
                """
            ).fetchone()

        total_papers += db_total
        successful_markdown += db_successful

    failed_or_none = total_papers - successful_markdown
    completion_percent = (
        100 * successful_markdown / total_papers
        if total_papers
        else 0.0
    )
    summary_rows.append(
        {
            "Source": source,
            "Databases": len(paths),
            "Papers": total_papers,
            "Successful Markdown": successful_markdown,
            "Failed / None": failed_or_none,
            "Completion (%)": completion_percent,
        }
    )

conversion_progress = pd.DataFrame(summary_rows)
overall_papers = int(conversion_progress["Papers"].sum())
overall_successful = int(
    conversion_progress["Successful Markdown"].sum()
)
overall_failed_or_none = overall_papers - overall_successful
overall_row = pd.DataFrame(
    [
        {
            "Source": "TOTAL",
            "Databases": int(conversion_progress["Databases"].sum()),
            "Papers": overall_papers,
            "Successful Markdown": overall_successful,
            "Failed / None": overall_failed_or_none,
            "Completion (%)": (
                100 * overall_successful / overall_papers
                if overall_papers
                else 0.0
            ),
        }
    ]
)
conversion_progress = pd.concat(
    [conversion_progress, overall_row], ignore_index=True
)

conversion_progress.style.format(
    {
        "Databases": "{:,}",
        "Papers": "{:,}",
        "Successful Markdown": "{:,}",
        "Failed / None": "{:,}",
        "Completion (%)": "{:.2f}%",
    }
).hide(axis="index")


Source,Databases,Papers,Successful Markdown,Failed / None,Completion (%)
arxiv,1,"5,388","5,387",1,99.98%
elsevier,1,"489,472","488,448","1,024",99.79%
europepmc,1,"185,447","185,447",0,100.00%
s2orc,1,"150,105","150,105",0,100.00%
springer_nature,1,"6,103","6,103",0,100.00%
wiley,36,"176,784","176,777",7,100.00%
TOTAL,41,"1,013,299","1,012,267","1,032",99.90%


In [46]:
# Show Wiley Markdown-conversion progress for each numbered part.
wiley_part_rows = []
for part_number, db_path in wiley_parts:
    database_uri = f"{db_path.as_uri()}?mode=ro"
    with closing(
        sqlite3.connect(database_uri, uri=True, timeout=60)
    ) as conn:
        conn.execute("PRAGMA query_only = ON")
        conn.execute("PRAGMA busy_timeout = 60000")
        part_total, part_successful = conn.execute(
            """
            SELECT
                COUNT(*),
                COALESCE(
                    SUM(CASE WHEN convert_md = 1 THEN 1 ELSE 0 END),
                    0
                )
            FROM papers
            """
        ).fetchone()

    part_failed_or_none = part_total - part_successful
    wiley_part_rows.append(
        {
            "Part": part_number,
            "Database": db_path.name,
            "Papers": part_total,
            "Successful Markdown": part_successful,
            "Failed / None": part_failed_or_none,
            "Completion (%)": (
                100 * part_successful / part_total
                if part_total
                else 0.0
            ),
        }
    )

wiley_part_progress = pd.DataFrame(wiley_part_rows)
wiley_part_progress.style.format(
    {
        "Part": "{:.0f}",
        "Papers": "{:,}",
        "Successful Markdown": "{:,}",
        "Failed / None": "{:,}",
        "Completion (%)": "{:.2f}%",
    }
).hide(axis="index").set_caption(
    "Wiley Markdown conversion progress by database part"
)


Part,Database,Papers,Successful Markdown,Failed / None,Completion (%)
1,wiley_track_part_1.db,"5,000","5,000",0,100.00%
2,wiley_track_part_2.db,"5,000","5,000",0,100.00%
3,wiley_track_part_3.db,"5,000","5,000",0,100.00%
4,wiley_track_part_4.db,"5,000","5,000",0,100.00%
5,wiley_track_part_5.db,"5,000","5,000",0,100.00%
6,wiley_track_part_6.db,"5,000","5,000",0,100.00%
7,wiley_track_part_7.db,"5,000","5,000",0,100.00%
8,wiley_track_part_8.db,"5,000","5,000",0,100.00%
9,wiley_track_part_9.db,"5,000","5,000",0,100.00%
10,wiley_track_part_10.db,"5,000","5,000",0,100.00%


In [47]:
# Print only Wiley database parts that are not fully converted.
incomplete_wiley_parts = wiley_part_progress.loc[
    wiley_part_progress["Failed / None"] > 0
].copy()

if incomplete_wiley_parts.empty:
    print("All Wiley database parts are complete.")
else:
    print("Incomplete Wiley database parts:")
    for row in incomplete_wiley_parts.to_dict(orient="records"):
        print(
            f"Part {row['Part']}: {row['Database']} - "
            f"{row['Failed / None']:,} failed or none "
            f"({row['Completion (%)']:.2f}% complete)"
        )


Incomplete Wiley database parts:
Part 12: wiley_track_part_12.db - 4 failed or none (99.92% complete)
Part 32: wiley_track_part_32.db - 2 failed or none (99.96% complete)
Part 33: wiley_track_part_33.db - 1 failed or none (99.98% complete)


In [ ]:
# # Merge all Wiley part databases back into wiley_track.db.
# # Run this only after all Wiley conversion jobs have stopped.
# from contextlib import closing
# from pathlib import Path
# import re
# import sqlite3

# process_dir = Path("../data/process").resolve()
# wiley_track_path = process_dir / "wiley_track.db"
# temporary_path = process_dir / "wiley_track.db.merge.tmp"
# part_pattern = re.compile(r"^wiley_track_part_(\d+)\.db$")

# if not wiley_track_path.is_file():
#     raise FileNotFoundError(
#         f"Original Wiley tracking database not found: {wiley_track_path}"
#     )

# numbered_parts = []
# for path in process_dir.glob("wiley_track_part_*.db"):
#     match = part_pattern.fullmatch(path.name)
#     if match is not None:
#         numbered_parts.append((int(match.group(1)), path))
# numbered_parts.sort(key=lambda item: item[0])

# if not numbered_parts:
#     raise FileNotFoundError(
#         f"No Wiley part databases found in {process_dir}"
#     )

# part_numbers = [part_number for part_number, _ in numbered_parts]
# expected_part_numbers = list(range(1, part_numbers[-1] + 1))
# if part_numbers != expected_part_numbers:
#     missing_parts = sorted(set(expected_part_numbers) - set(part_numbers))
#     raise RuntimeError(f"Missing Wiley database part(s): {missing_parts}")

# with closing(
#     sqlite3.connect(
#         f"{wiley_track_path.as_uri()}?mode=ro",
#         uri=True,
#         timeout=60,
#     )
# ) as original_conn:
#     original_conn.execute("PRAGMA query_only = ON")
#     original_conn.execute("PRAGMA busy_timeout = 60000")
#     original_count = original_conn.execute(
#         "SELECT COUNT(*) FROM papers"
#     ).fetchone()[0]

# if temporary_path.exists():
#     temporary_path.unlink()

# copied_count = 0
# successful_count = 0
# try:
#     with closing(
#         sqlite3.connect(
#             temporary_path, timeout=60, isolation_level=None
#         )
#     ) as merged_conn:
#         merged_conn.execute("PRAGMA busy_timeout = 60000")
#         merged_conn.execute("BEGIN IMMEDIATE")
#         merged_conn.execute(
#             """
#             CREATE TABLE papers (
#                 paper_id INTEGER PRIMARY KEY,
#                 paper_uid TEXT NOT NULL UNIQUE,
#                 canonical_source TEXT NOT NULL,
#                 convert_md INTEGER DEFAULT NULL
#             )
#             """
#         )

#         for part_number, part_path in numbered_parts:
#             part_uri = f"{part_path.as_uri()}?mode=ro"
#             with closing(
#                 sqlite3.connect(part_uri, uri=True, timeout=60)
#             ) as part_conn:
#                 part_conn.execute("PRAGMA query_only = ON")
#                 part_conn.execute("PRAGMA busy_timeout = 60000")

#                 columns = {
#                     row[1] for row in part_conn.execute(
#                         "PRAGMA table_info(papers)"
#                     )
#                 }
#                 required_columns = {
#                     "paper_id",
#                     "paper_uid",
#                     "canonical_source",
#                     "convert_md",
#                 }
#                 if not required_columns <= columns:
#                     missing_columns = sorted(required_columns - columns)
#                     raise ValueError(
#                         f"{part_path.name} is missing column(s): "
#                         + ", ".join(missing_columns)
#                     )

#                 invalid_source_count = part_conn.execute(
#                     """
#                     SELECT COUNT(*)
#                     FROM papers
#                     WHERE canonical_source != 'wiley'
#                     """
#                 ).fetchone()[0]
#                 if invalid_source_count:
#                     raise ValueError(
#                         f"{part_path.name} contains "
#                         f"{invalid_source_count:,} non-Wiley row(s)"
#                     )

#                 rows = part_conn.execute(
#                     """
#                     SELECT
#                         paper_id, paper_uid, canonical_source, convert_md
#                     FROM papers
#                     ORDER BY paper_id
#                     """
#                 ).fetchall()

#             if len(rows) > 5_000:
#                 raise ValueError(
#                     f"{part_path.name} contains more than 5,000 rows"
#                 )
#             merged_conn.executemany(
#                 """
#                 INSERT INTO papers (
#                     paper_id, paper_uid, canonical_source, convert_md
#                 )
#                 VALUES (?, ?, ?, ?)
#                 """,
#                 rows,
#             )
#             copied_count += len(rows)
#             successful_count += sum(row[3] == 1 for row in rows)

#         merged_count = merged_conn.execute(
#             "SELECT COUNT(*) FROM papers"
#         ).fetchone()[0]
#         if merged_count != copied_count:
#             raise RuntimeError(
#                 f"Expected {copied_count:,} merged rows but found "
#                 f"{merged_count:,}"
#             )
#         if merged_count != original_count:
#             raise RuntimeError(
#                 f"Parts contain {merged_count:,} rows but the original "
#                 f"tracker contains {original_count:,}"
#             )
#         merged_conn.commit()

#     # Confirm that part metadata exactly matches the original tracker.
#     with closing(sqlite3.connect(temporary_path, timeout=60)) as check_conn:
#         check_conn.execute("PRAGMA busy_timeout = 60000")
#         check_conn.execute(
#             "ATTACH DATABASE ? AS original",
#             (str(wiley_track_path),),
#         )
#         missing_from_merge = check_conn.execute(
#             """
#             SELECT COUNT(*)
#             FROM (
#                 SELECT paper_id, paper_uid, canonical_source
#                 FROM original.papers
#                 EXCEPT
#                 SELECT paper_id, paper_uid, canonical_source
#                 FROM main.papers
#             )
#             """
#         ).fetchone()[0]
#         extra_in_merge = check_conn.execute(
#             """
#             SELECT COUNT(*)
#             FROM (
#                 SELECT paper_id, paper_uid, canonical_source
#                 FROM main.papers
#                 EXCEPT
#                 SELECT paper_id, paper_uid, canonical_source
#                 FROM original.papers
#             )
#             """
#         ).fetchone()[0]
#         integrity_result = check_conn.execute(
#             "PRAGMA main.integrity_check"
#         ).fetchone()[0]
#         check_conn.execute("DETACH DATABASE original")

#     if missing_from_merge or extra_in_merge:
#         raise RuntimeError(
#             "Merged metadata does not match the original tracker: "
#             f"missing={missing_from_merge:,}, extra={extra_in_merge:,}"
#         )
#     if integrity_result != "ok":
#         raise RuntimeError(
#             f"Merged database failed integrity check: {integrity_result}"
#         )

#     # Atomic replacement: the original remains untouched until this point.
#     temporary_path.replace(wiley_track_path)
# except Exception:
#     if temporary_path.exists():
#         temporary_path.unlink()
#     raise

# failed_or_none_count = copied_count - successful_count
# print(f"Updated {wiley_track_path}")
# print(f"Wiley parts merged: {len(numbered_parts)}")
# print(f"Total rows: {copied_count:,}")
# print(f"Successful Markdown: {successful_count:,}")
# print(f"Failed / None: {failed_or_none_count:,}")


Updated /Users/kevinge/Work/Data Extraction/synth_extract/data/process/wiley_track.db
Wiley parts merged: 36
Total rows: 176,784
Successful Markdown: 176,777
Failed / None: 7


In [49]:
# Summarize progress using one main tracking database per source.
from contextlib import closing
from pathlib import Path
import sqlite3

import pandas as pd

process_dir = Path("../data/process").resolve()
sources = [
    "arxiv",
    "elsevier",
    "europepmc",
    "s2orc",
    "springer_nature",
    "wiley",
]

if not process_dir.is_dir():
    raise FileNotFoundError(f"Process directory not found: {process_dir}")

summary_rows = []
for source in sources:
    db_path = process_dir / f"{source}_track.db"
    if not db_path.is_file():
        raise FileNotFoundError(
            f"Tracking database not found: {db_path}"
        )

    database_uri = f"{db_path.as_uri()}?mode=ro"
    with closing(
        sqlite3.connect(database_uri, uri=True, timeout=60)
    ) as conn:
        conn.execute("PRAGMA query_only = ON")
        conn.execute("PRAGMA busy_timeout = 60000")

        table_exists = conn.execute(
            """
            SELECT 1
            FROM sqlite_master
            WHERE type = 'table' AND name = 'papers'
            """
        ).fetchone()
        if table_exists is None:
            raise ValueError(f"No papers table in {db_path}")

        columns = {
            row[1] for row in conn.execute(
                "PRAGMA table_info(papers)"
            )
        }
        if "convert_md" not in columns:
            raise ValueError(f"No convert_md column in {db_path}")

        total_papers, successful_markdown = conn.execute(
            """
            SELECT
                COUNT(*),
                COALESCE(
                    SUM(CASE WHEN convert_md = 1 THEN 1 ELSE 0 END),
                    0
                )
            FROM papers
            """
        ).fetchone()

    failed_or_none = total_papers - successful_markdown
    summary_rows.append(
        {
            "Source": source,
            "Database": db_path.name,
            "Papers": total_papers,
            "Successful Markdown": successful_markdown,
            "Failed / None": failed_or_none,
            "Completion (%)": (
                100 * successful_markdown / total_papers
                if total_papers
                else 0.0
            ),
        }
    )

main_db_progress = pd.DataFrame(summary_rows)
overall_papers = int(main_db_progress["Papers"].sum())
overall_successful = int(
    main_db_progress["Successful Markdown"].sum()
)
overall_row = pd.DataFrame(
    [
        {
            "Source": "TOTAL",
            "Database": "All main tracking databases",
            "Papers": overall_papers,
            "Successful Markdown": overall_successful,
            "Failed / None": overall_papers - overall_successful,
            "Completion (%)": (
                100 * overall_successful / overall_papers
                if overall_papers
                else 0.0
            ),
        }
    ]
)
main_db_progress = pd.concat(
    [main_db_progress, overall_row], ignore_index=True
)

main_db_progress.style.format(
    {
        "Papers": "{:,}",
        "Successful Markdown": "{:,}",
        "Failed / None": "{:,}",
        "Completion (%)": "{:.2f}%",
    }
).hide(axis="index")


Source,Database,Papers,Successful Markdown,Failed / None,Completion (%)
arxiv,arxiv_track.db,"5,388","5,387",1,99.98%
elsevier,elsevier_track.db,"489,472","488,448","1,024",99.79%
europepmc,europepmc_track.db,"185,447","185,447",0,100.00%
s2orc,s2orc_track.db,"150,105","150,105",0,100.00%
springer_nature,springer_nature_track.db,"6,103","6,103",0,100.00%
wiley,wiley_track.db,"176,784","176,777",7,100.00%
TOTAL,All main tracking databases,"1,013,299","1,012,267","1,032",99.90%


In [53]:
import synth_extract.utils.sql_helpers as sql

In [54]:
sql.get_table_schema("/Users/kevinge/Work/Data Extraction/synth_extract/data/central_papers.db", "papers")

,cid,name,type,notnull,dflt_value,pk
0,0,paper_id,INTEGER,0,None,1
1,1,paper_uid,TEXT,1,None,0
2,2,identifier_type,TEXT,1,None,0
3,3,identifier_value,TEXT,1,None,0
4,4,doi,TEXT,0,None,0
5,5,arxiv_id,TEXT,0,None,0
6,6,pmcid,TEXT,0,None,0
7,7,title,TEXT,0,None,0
8,8,abstract,TEXT,0,None,0
9,9,sources,TEXT,1,None,0


In [ ]:
# # Build a compact table containing only successfully downloaded papers.
# from contextlib import closing
# from pathlib import Path
# import sqlite3

# central_db_path = Path("../data/central_papers.db").resolve()
# excluded_columns = {
#     "abstract",
#     "failure_history",
#     "last_attempted_source",
#     "last_attempted_at",
#     "last_error",
#     "downloaded_from",
#     "downloaded_at",
#     "review_note",
#     "created_at",
#     "updated_at",
# }
# retained_columns = [
#     "paper_id",
#     "paper_uid",
#     "identifier_type",
#     "identifier_value",
#     "doi",
#     "arxiv_id",
#     "pmcid",
#     "title",
#     "sources",
#     "source_count",
#     "source_order",
#     "canonical_source",
#     "canonical_source_position",
#     "download_status",
#     "attempt_count",
#     "attempted_sources",
#     "fulltext_path",
#     "fulltext_format",
# ]

# if not central_db_path.is_file():
#     raise FileNotFoundError(
#         f"Central database not found: {central_db_path}"
#     )

# with closing(
#     sqlite3.connect(central_db_path, timeout=60, isolation_level=None)
# ) as conn:
#     conn.execute("PRAGMA busy_timeout = 60000")

#     source_table_exists = conn.execute(
#         """
#         SELECT 1
#         FROM sqlite_master
#         WHERE type = 'table' AND name = 'papers'
#         """
#     ).fetchone()
#     if source_table_exists is None:
#         raise ValueError("Central database has no papers table")

#     source_columns = [
#         row[1] for row in conn.execute("PRAGMA table_info(papers)")
#     ]
#     expected_retained_columns = [
#         column for column in source_columns
#         if column not in excluded_columns
#     ]
#     if retained_columns != expected_retained_columns:
#         raise RuntimeError(
#             "The papers schema changed. Expected retained columns "
#             f"{expected_retained_columns}, but the cell defines "
#             f"{retained_columns}"
#         )

#     try:
#         conn.execute("BEGIN IMMEDIATE")
#         source_success_count = conn.execute(
#             """
#             SELECT COUNT(*)
#             FROM papers
#             WHERE download_status = 'success'
#             """
#         ).fetchone()[0]

#         # Rebuild the derived table so rerunning the cell refreshes it.
#         conn.execute("DROP TABLE IF EXISTS downloaded_papers")
#         conn.execute(
#             """
#             CREATE TABLE downloaded_papers (
#                 paper_id INTEGER PRIMARY KEY,
#                 paper_uid TEXT NOT NULL UNIQUE,
#                 identifier_type TEXT NOT NULL
#                     CHECK (identifier_type IN ('doi', 'arxiv_id', 'pmcid')),
#                 identifier_value TEXT NOT NULL UNIQUE,
#                 doi TEXT,
#                 arxiv_id TEXT,
#                 pmcid TEXT,
#                 title TEXT,
#                 sources TEXT NOT NULL,
#                 source_count INTEGER NOT NULL,
#                 source_order TEXT NOT NULL,
#                 canonical_source TEXT NOT NULL,
#                 canonical_source_position INTEGER NOT NULL DEFAULT 1,
#                 download_status TEXT NOT NULL
#                     CHECK (download_status = 'success'),
#                 attempt_count INTEGER NOT NULL DEFAULT 0,
#                 attempted_sources TEXT NOT NULL DEFAULT '[]',
#                 fulltext_path TEXT,
#                 fulltext_format TEXT,
#                 CHECK (
#                     (identifier_type = 'doi' AND doi IS NOT NULL)
#                     OR
#                     (identifier_type = 'arxiv_id' AND arxiv_id IS NOT NULL)
#                     OR
#                     (identifier_type = 'pmcid' AND identifier_value IS NOT NULL)
#                 )
#             )
#             """
#         )
#         conn.execute(
#             """
#             INSERT INTO downloaded_papers (
#                 paper_id,
#                 paper_uid,
#                 identifier_type,
#                 identifier_value,
#                 doi,
#                 arxiv_id,
#                 pmcid,
#                 title,
#                 sources,
#                 source_count,
#                 source_order,
#                 canonical_source,
#                 canonical_source_position,
#                 download_status,
#                 attempt_count,
#                 attempted_sources,
#                 fulltext_path,
#                 fulltext_format
#             )
#             SELECT
#                 paper_id,
#                 paper_uid,
#                 identifier_type,
#                 identifier_value,
#                 doi,
#                 arxiv_id,
#                 pmcid,
#                 title,
#                 sources,
#                 source_count,
#                 source_order,
#                 canonical_source,
#                 canonical_source_position,
#                 download_status,
#                 attempt_count,
#                 attempted_sources,
#                 fulltext_path,
#                 fulltext_format
#             FROM papers
#             WHERE download_status = 'success'
#             """
#         )

#         conn.execute(
#             """
#             CREATE UNIQUE INDEX idx_downloaded_papers_identifier
#             ON downloaded_papers(identifier_type, identifier_value)
#             """
#         )
#         conn.execute(
#             """
#             CREATE UNIQUE INDEX idx_downloaded_papers_doi
#             ON downloaded_papers(doi)
#             WHERE doi IS NOT NULL
#             """
#         )
#         conn.execute(
#             """
#             CREATE UNIQUE INDEX idx_downloaded_papers_arxiv_only
#             ON downloaded_papers(arxiv_id)
#             WHERE identifier_type = 'arxiv_id' AND arxiv_id IS NOT NULL
#             """
#         )
#         conn.execute(
#             """
#             CREATE INDEX idx_downloaded_papers_canonical_source
#             ON downloaded_papers(canonical_source)
#             """
#         )

#         copied_count = conn.execute(
#             "SELECT COUNT(*) FROM downloaded_papers"
#         ).fetchone()[0]
#         invalid_status_count = conn.execute(
#             """
#             SELECT COUNT(*)
#             FROM downloaded_papers
#             WHERE download_status != 'success'
#             """
#         ).fetchone()[0]
#         downloaded_columns = [
#             row[1] for row in conn.execute(
#                 "PRAGMA table_info(downloaded_papers)"
#             )
#         ]

#         if copied_count != source_success_count:
#             raise RuntimeError(
#                 f"Expected {source_success_count:,} successful rows, "
#                 f"but copied {copied_count:,}"
#             )
#         if invalid_status_count:
#             raise RuntimeError(
#                 f"Copied {invalid_status_count:,} non-success row(s)"
#             )
#         if downloaded_columns != retained_columns:
#             raise RuntimeError(
#                 "downloaded_papers has unexpected columns: "
#                 f"{downloaded_columns}"
#             )

#         conn.commit()
#     except Exception:
#         conn.rollback()
#         raise

# print(f"Created downloaded_papers in {central_db_path}")
# print(f"Successful rows copied: {copied_count:,}")
# print(f"Columns retained: {len(retained_columns)}")
# print(f"Columns excluded: {len(excluded_columns)}")


Created downloaded_papers in /Users/kevinge/Work/Data Extraction/synth_extract/data/central_papers.db
Successful rows copied: 1,013,299
Columns retained: 18
Columns excluded: 10


In [ ]:
# # Add an empty Markdown-conversion status to downloaded_papers.
# from contextlib import closing
# from pathlib import Path
# import sqlite3

# central_db_path = Path("../data/central_papers.db").resolve()
# if not central_db_path.is_file():
#     raise FileNotFoundError(
#         f"Central database not found: {central_db_path}"
#     )

# with closing(
#     sqlite3.connect(central_db_path, timeout=60, isolation_level=None)
# ) as conn:
#     conn.execute("PRAGMA busy_timeout = 60000")

#     table_exists = conn.execute(
#         """
#         SELECT 1
#         FROM sqlite_master
#         WHERE type = 'table' AND name = 'downloaded_papers'
#         """
#     ).fetchone()
#     if table_exists is None:
#         raise ValueError(
#             "Central database has no downloaded_papers table"
#         )

#     try:
#         conn.execute("BEGIN IMMEDIATE")
#         column_info = {
#             row[1]: row
#             for row in conn.execute(
#                 "PRAGMA table_info(downloaded_papers)"
#             )
#         }

#         column_created = "markdown_status" not in column_info
#         if column_created:
#             conn.execute(
#                 """
#                 ALTER TABLE downloaded_papers
#                 ADD COLUMN markdown_status INTEGER DEFAULT NULL
#                 """
#             )
#         else:
#             declared_type = (column_info["markdown_status"][2] or "").upper()
#             if declared_type != "INTEGER":
#                 raise ValueError(
#                     "Existing markdown_status column is not INTEGER: "
#                     f"{declared_type!r}"
#                 )

#         total_rows, populated_rows = conn.execute(
#             """
#             SELECT
#                 COUNT(*),
#                 COALESCE(
#                     SUM(CASE WHEN markdown_status IS NOT NULL THEN 1 ELSE 0 END),
#                     0
#                 )
#             FROM downloaded_papers
#             """
#         ).fetchone()
#         if populated_rows:
#             raise RuntimeError(
#                 f"markdown_status is already populated for "
#                 f"{populated_rows:,} row(s); no values were changed"
#             )

#         conn.commit()
#     except Exception:
#         conn.rollback()
#         raise

# action = "Created" if column_created else "Verified existing"
# print(f"{action} downloaded_papers.markdown_status")
# print(f"Rows with NULL markdown_status: {total_rows:,}")
# print("Rows with populated markdown_status: 0")


Created downloaded_papers.markdown_status
Rows with NULL markdown_status: 1,013,299
Rows with populated markdown_status: 0


In [ ]:
# # Populate downloaded_papers.markdown_status from the main track databases.
# # Run this after conversion jobs stop and after merging the Wiley parts.
# from contextlib import closing
# from pathlib import Path
# import sqlite3

# import pandas as pd

# central_db_path = Path("../data/central_papers.db").resolve()
# process_dir = Path("../data/process").resolve()
# sources = [
#     "arxiv",
#     "elsevier",
#     "europepmc",
#     "s2orc",
#     "springer_nature",
#     "wiley",
# ]
# track_paths = {
#     source: process_dir / f"{source}_track.db"
#     for source in sources
# }

# if not central_db_path.is_file():
#     raise FileNotFoundError(
#         f"Central database not found: {central_db_path}"
#     )
# for source, track_path in track_paths.items():
#     if not track_path.is_file():
#         raise FileNotFoundError(
#             f"{source} tracking database not found: {track_path}"
#         )

# attached_aliases = []
# with closing(
#     sqlite3.connect(central_db_path, timeout=60, isolation_level=None)
# ) as conn:
#     conn.execute("PRAGMA busy_timeout = 60000")

#     downloaded_columns = {
#         row[1] for row in conn.execute(
#             "PRAGMA table_info(downloaded_papers)"
#         )
#     }
#     required_downloaded_columns = {
#         "paper_id",
#         "canonical_source",
#         "markdown_status",
#     }
#     missing_downloaded_columns = sorted(
#         required_downloaded_columns - downloaded_columns
#     )
#     if missing_downloaded_columns:
#         raise ValueError(
#             "downloaded_papers is missing column(s): "
#             + ", ".join(missing_downloaded_columns)
#         )

#     try:
#         # Attach and validate every tracker before updating any statuses.
#         for source, track_path in track_paths.items():
#             alias = f"track_{source}"
#             conn.execute(
#                 f"ATTACH DATABASE ? AS {alias}",
#                 (str(track_path),),
#             )
#             attached_aliases.append(alias)

#             track_columns = {
#                 row[1] for row in conn.execute(
#                     f"PRAGMA {alias}.table_info(papers)"
#                 )
#             }
#             required_track_columns = {
#                 "paper_id",
#                 "canonical_source",
#                 "convert_md",
#             }
#             missing_track_columns = sorted(
#                 required_track_columns - track_columns
#             )
#             if missing_track_columns:
#                 raise ValueError(
#                     f"{track_path.name} is missing column(s): "
#                     + ", ".join(missing_track_columns)
#                 )

#             track_count = conn.execute(
#                 f"SELECT COUNT(*) FROM {alias}.papers"
#             ).fetchone()[0]
#             downloaded_count = conn.execute(
#                 """
#                 SELECT COUNT(*)
#                 FROM downloaded_papers
#                 WHERE canonical_source = ?
#                 """,
#                 (source,),
#             ).fetchone()[0]
#             invalid_track_source_count = conn.execute(
#                 f"""
#                 SELECT COUNT(*)
#                 FROM {alias}.papers
#                 WHERE canonical_source != ?
#                 """,
#                 (source,),
#             ).fetchone()[0]
#             missing_downloaded_rows = conn.execute(
#                 f"""
#                 SELECT COUNT(*)
#                 FROM {alias}.papers AS track
#                 LEFT JOIN downloaded_papers AS downloaded
#                     ON downloaded.paper_id = track.paper_id
#                 WHERE downloaded.paper_id IS NULL
#                    OR downloaded.canonical_source != ?
#                 """,
#                 (source,),
#             ).fetchone()[0]
#             missing_track_rows = conn.execute(
#                 f"""
#                 SELECT COUNT(*)
#                 FROM downloaded_papers AS downloaded
#                 LEFT JOIN {alias}.papers AS track
#                     ON track.paper_id = downloaded.paper_id
#                 WHERE downloaded.canonical_source = ?
#                   AND track.paper_id IS NULL
#                 """,
#                 (source,),
#             ).fetchone()[0]

#             if track_count != downloaded_count:
#                 raise RuntimeError(
#                     f"{source}: tracker has {track_count:,} rows but "
#                     f"downloaded_papers has {downloaded_count:,}"
#                 )
#             if invalid_track_source_count:
#                 raise RuntimeError(
#                     f"{source}: tracker contains "
#                     f"{invalid_track_source_count:,} row(s) for another source"
#                 )
#             if missing_downloaded_rows or missing_track_rows:
#                 raise RuntimeError(
#                     f"{source}: paper_id coverage mismatch; "
#                     f"missing from downloaded_papers={missing_downloaded_rows:,}, "
#                     f"missing from tracker={missing_track_rows:,}"
#                 )

#         conn.execute("BEGIN IMMEDIATE")
#         for source in sources:
#             alias = f"track_{source}"
#             conn.execute(
#                 f"""
#                 UPDATE downloaded_papers AS downloaded
#                 SET markdown_status = COALESCE(
#                     (
#                         SELECT track.convert_md
#                         FROM {alias}.papers AS track
#                         WHERE track.paper_id = downloaded.paper_id
#                     ),
#                     0
#                 )
#                 WHERE downloaded.canonical_source = ?
#                 """,
#                 (source,),
#             )

#             mismatched_status_count = conn.execute(
#                 f"""
#                 SELECT COUNT(*)
#                 FROM downloaded_papers AS downloaded
#                 JOIN {alias}.papers AS track
#                     ON track.paper_id = downloaded.paper_id
#                 WHERE NOT (
#                     downloaded.markdown_status
#                     IS COALESCE(track.convert_md, 0)
#                 )
#                 """
#             ).fetchone()[0]
#             if mismatched_status_count:
#                 raise RuntimeError(
#                     f"{source}: {mismatched_status_count:,} copied "
#                     "Markdown status value(s) do not match the tracker"
#                 )

#         conn.commit()
#     except Exception:
#         if conn.in_transaction:
#             conn.rollback()
#         raise
#     finally:
#         for alias in reversed(attached_aliases):
#             conn.execute(f"DETACH DATABASE {alias}")

#     status_rows = conn.execute(
#         """
#         SELECT
#             canonical_source,
#             COUNT(*) AS papers,
#             COALESCE(
#                 SUM(CASE WHEN markdown_status = 1 THEN 1 ELSE 0 END),
#                 0
#             ) AS completed,
#             COALESCE(
#                 SUM(CASE WHEN markdown_status = 0 THEN 1 ELSE 0 END),
#                 0
#             ) AS failed_or_none,
#             COALESCE(
#                 SUM(
#                     CASE
#                         WHEN markdown_status IS NULL
#                           OR markdown_status NOT IN (0, 1) THEN 1
#                         ELSE 0
#                     END
#                 ),
#                 0
#             ) AS other_status
#         FROM downloaded_papers
#         GROUP BY canonical_source
#         ORDER BY canonical_source
#         """
#     ).fetchall()

# markdown_status_summary = pd.DataFrame(
#     status_rows,
#     columns=[
#         "Source",
#         "Papers",
#         "Completed Markdown",
#         "Incomplete Markdown",
#         "Other Status",
#     ],
# )
# markdown_status_summary.style.format(
#     {
#         "Papers": "{:,}",
#         "Completed Markdown": "{:,}",
#         "Incomplete Markdown": "{:,}",
#         "Other Status": "{:,}",
#     }
# ).hide(axis="index")


Source,Papers,Completed Markdown,Incomplete Markdown,Other Status
arxiv,"5,388","5,387",1,0
elsevier,"489,472","488,448","1,024",0
europepmc,"185,447","185,447",0,0
s2orc,"150,105","150,105",0,0
springer_nature,"6,103","6,103",0,0
wiley,"176,784","176,777",7,0


New Track table

In [ ]:
# # Report Markdown status per source from downloaded_papers.
# from contextlib import closing
# from pathlib import Path
# import sqlite3

# import pandas as pd

# central_db_path = Path("../data/central_papers.db").resolve()
# if not central_db_path.is_file():
#     raise FileNotFoundError(
#         f"Central database not found: {central_db_path}"
#     )

# database_uri = f"{central_db_path.as_uri()}?mode=ro"
# with closing(
#     sqlite3.connect(database_uri, uri=True, timeout=60)
# ) as conn:
#     conn.execute("PRAGMA query_only = ON")
#     conn.execute("PRAGMA busy_timeout = 60000")

#     required_columns = {"canonical_source", "markdown_status"}
#     available_columns = {
#         row[1] for row in conn.execute(
#             "PRAGMA table_info(downloaded_papers)"
#         )
#     }
#     missing_columns = sorted(required_columns - available_columns)
#     if missing_columns:
#         raise ValueError(
#             "downloaded_papers is missing column(s): "
#             + ", ".join(missing_columns)
#         )

#     invalid_status_count = conn.execute(
#         """
#         SELECT COUNT(*)
#         FROM downloaded_papers
#         WHERE markdown_status IS NULL
#            OR markdown_status NOT IN (0, 1)
#         """
#     ).fetchone()[0]
#     if invalid_status_count:
#         raise ValueError(
#             f"downloaded_papers contains {invalid_status_count:,} "
#             "unset or unexpected Markdown status value(s)"
#         )

#     source_markdown_status = pd.read_sql_query(
#         """
#         SELECT
#             canonical_source AS Source,
#             COUNT(*) AS Papers,
#             SUM(
#                 CASE WHEN markdown_status = 1 THEN 1 ELSE 0 END
#             ) AS "Successful Markdown",
#             SUM(
#                 CASE WHEN markdown_status = 0 THEN 1 ELSE 0 END
#             ) AS "Failed / Incomplete"
#         FROM downloaded_papers
#         GROUP BY canonical_source
#         ORDER BY canonical_source
#         """,
#         conn,
#     )

# source_markdown_status["Completion (%)"] = (
#     100
#     * source_markdown_status["Successful Markdown"]
#     / source_markdown_status["Papers"]
# )
# total_papers = int(source_markdown_status["Papers"].sum())
# total_successful = int(
#     source_markdown_status["Successful Markdown"].sum()
# )
# total_failed = int(
#     source_markdown_status["Failed / Incomplete"].sum()
# )
# total_row = pd.DataFrame(
#     [
#         {
#             "Source": "TOTAL",
#             "Papers": total_papers,
#             "Successful Markdown": total_successful,
#             "Failed / Incomplete": total_failed,
#             "Completion (%)": (
#                 100 * total_successful / total_papers
#                 if total_papers
#                 else 0.0
#             ),
#         }
#     ]
# )
# source_markdown_status = pd.concat(
#     [source_markdown_status, total_row], ignore_index=True
# )

# source_markdown_status.style.format(
#     {
#         "Papers": "{:,}",
#         "Successful Markdown": "{:,}",
#         "Failed / Incomplete": "{:,}",
#         "Completion (%)": "{:.2f}%",
#     }
# ).hide(axis="index")


Source,Papers,Successful Markdown,Failed / Incomplete,Completion (%)
arxiv,"5,388","5,387",1,99.98%
elsevier,"489,472","488,448","1,024",99.79%
europepmc,"185,447","185,447",0,100.00%
s2orc,"150,105","150,105",0,100.00%
springer_nature,"6,103","6,103",0,100.00%
wiley,"176,784","176,777",7,100.00%
TOTAL,"1,013,299","1,012,267","1,032",99.90%
